# Notebook 01 — Data Cleaning & Preprocessing
**Author:** Pasindu  
**Role:** Data Engineer  
**Task:** Load raw CSV → Upload to Hadoop HDFS → Clean & Preprocess → Save cleaned data to HDFS & locally

---
> **Run this notebook FIRST before any other notebook.**  
> All other notebooks depend on the cleaned Parquet saved here.

## 1. Load Packages

In [9]:
import os
import subprocess

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, TimestampType, DoubleType
)

print("Packages loaded successfully.")

Packages loaded successfully.


## 2. Start Spark Session

In [10]:
spark = SparkSession.builder \
    .appName("RetailDataCleaning") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("Spark session started.")

Spark version: 4.1.1
Spark session started.


## 3. Upload Raw Dataset to Hadoop HDFS

We first create the required directories in HDFS, then upload the raw CSV file.  
This is the **only time** we touch the raw CSV — all other notebooks read from HDFS.

In [11]:
# ── Path to the raw CSV on your local machine ────────────────────────────────
LOCAL_RAW_CSV = "/Users/pasindumalinda/Project-04/ecommerce-recommendation-spark/data/raw/online_retail_II.csv"   # adjust if different

# ── HDFS paths ───────────────────────────────────────────────────────────────
HDFS_RAW_DIR     = "hdfs:///retail/raw/"
HDFS_CLEANED_DIR = "hdfs:///retail/cleaned/"

# ── Local output path (for GitHub) ──────────────────────────────────────────
LOCAL_CLEANED_DIR = "output/cleaned_data/"
os.makedirs(LOCAL_CLEANED_DIR, exist_ok=True)

# ── Create HDFS directories ──────────────────────────────────────────────────
subprocess.run(["hdfs", "dfs", "-mkdir", "-p", "/retail/raw"],     check=True)
subprocess.run(["hdfs", "dfs", "-mkdir", "-p", "/retail/cleaned"], check=True)

# ── Upload raw CSV to HDFS ────────────────────────────────────────────────────
subprocess.run(
    ["hdfs", "dfs", "-put", "-f", LOCAL_RAW_CSV, "/retail/raw/"],
    check=True
)

print(f"Raw CSV uploaded to HDFS: {HDFS_RAW_DIR}")

2026-04-26 09:56:04,120 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-04-26 09:56:04,686 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-04-26 09:56:05,166 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Raw CSV uploaded to HDFS: hdfs:///retail/raw/


## 4. Load Raw Dataset from HDFS

In [12]:
df_raw = spark.read.csv(
    "hdfs://localhost:8020/retail/raw/online_retail_II.csv",
    header=True,
    inferSchema=True
)

df_raw = df_raw \
    .withColumnRenamed("Customer ID", "CustomerID") \
    .withColumnRenamed("Price", "UnitPrice")

raw_count = df_raw.count()
print(f"Raw record count: {raw_count:,}")

Raw record count: 1,067,371


## 5. Explore the Raw Data

In [13]:
# Schema — column names and data types
df_raw.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [14]:
# Preview first 5 rows
df_raw.show(5, truncate=False)

+-------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95     |13085.0   |United Kingdom|
|489434 |79323P   |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75     |13085.0   |United Kingdom|
|489434 |79323W   | WHITE CHERRY LIGHTS               |12      |2009-12-01 07:45:00|6.75     |13085.0   |United Kingdom|
|489434 |22041    |"RECORD FRAME 7"" SINGLE SIZE "    |48      |2009-12-01 07:45:00|2.1      |13085.0   |United Kingdom|
|489434 |21232    |STRAWBERRY CERAMIC TRINKET BOX     |24      |2009-12-01 07:45:00|1.25     |13085.0   |United Kingdom|
+-------+---------+-------------

In [15]:
# Basic statistics for numeric columns
df_raw.select("Quantity", "UnitPrice", "CustomerID").describe().show()

+-------+------------------+------------------+------------------+
|summary|          Quantity|         UnitPrice|        CustomerID|
+-------+------------------+------------------+------------------+
|  count|           1067371|           1067371|            824364|
|   mean|   9.9388984711033| 4.649387727415796| 15324.63850435002|
| stddev|172.70579407675396|123.55305872146369|1697.4644503793133|
|    min|            -80995|         -53594.36|           12346.0|
|    max|             80995|           38970.0|           18287.0|
+-------+------------------+------------------+------------------+



In [16]:
# Count null values in each column
print("Null counts per column:")
df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

Null counts per column:
+-------+---------+-----------+--------+-----------+---------+----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+-------+---------+-----------+--------+-----------+---------+----------+-------+
|      0|        0|       4382|       0|          0|        0|    243007|      0|
+-------+---------+-----------+--------+-----------+---------+----------+-------+



## 6. Step-by-Step Data Cleaning

We apply the following cleaning steps in order:
1. Remove rows with null CustomerID or Description
2. Remove cancelled orders (Invoice starts with 'C')
3. Remove rows with Quantity ≤ 0 or UnitPrice ≤ 0
4. Remove duplicate rows
5. Fix the InvoiceDate column type
6. Add calculated columns

In [17]:
# ── Step 1: Remove rows where CustomerID or Description is null ──────────────
df = df_raw.dropna(subset=["CustomerID", "Description", "Invoice"])
print(f"After removing nulls: {df.count():,} rows  (removed {raw_count - df.count():,})")

After removing nulls: 824,364 rows  (removed 243,007)


In [18]:
# ── Step 2: Remove cancelled orders (Invoice starts with 'C') ─────────────────
df = df.filter(~F.col("Invoice").startswith("C"))
print(f"After removing cancellations: {df.count():,} rows")

After removing cancellations: 805,620 rows


In [19]:
# ── Step 3: Remove negative/zero Quantity and UnitPrice ───────────────────────
df = df.filter((F.col("Quantity") > 0) & (F.col("UnitPrice") > 0))
print(f"After removing invalid quantities/prices: {df.count():,} rows")

After removing invalid quantities/prices: 805,549 rows


In [20]:
# ── Step 4: Remove exact duplicate rows ───────────────────────────────────────
before_dedup = df.count()
df = df.dropDuplicates()
print(f"After deduplication: {df.count():,} rows  (removed {before_dedup - df.count():,} duplicates)")

After deduplication: 779,425 rows  (removed 26,124 duplicates)


In [21]:
# ── Step 5: Parse InvoiceDate and extract date parts ─────────────────────────
df = df.withColumn(
    "InvoiceDate",
    F.to_timestamp(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss")
)

# Extract Year, Month, Day, Hour for analysis
df = df \
    .withColumn("Year",      F.year("InvoiceDate")) \
    .withColumn("Month",     F.month("InvoiceDate")) \
    .withColumn("DayOfWeek", F.dayofweek("InvoiceDate")) \
    .withColumn("Hour",      F.hour("InvoiceDate")) \
    .withColumn("YearMonth", F.date_format("InvoiceDate", "yyyy-MM"))

print("Date columns added: Year, Month, DayOfWeek, Hour, YearMonth")

Date columns added: Year, Month, DayOfWeek, Hour, YearMonth


In [22]:
# ── Step 6: Add TotalPrice column (Quantity × UnitPrice) ─────────────────────
df = df.withColumn(
    "TotalPrice",
    F.round(F.col("Quantity") * F.col("UnitPrice"), 2)
)

# Cast CustomerID to integer for cleaner joins
df = df.withColumn("CustomerID", F.col("CustomerID").cast("integer"))

print("TotalPrice column added.")
print(f"Final cleaned record count: {df.count():,}")

TotalPrice column added.


Final cleaned record count: 779,425


## 7. Verify Cleaned Data

In [23]:
# Check the final schema
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- YearMonth: string (nullable = true)
 |-- TotalPrice: double (nullable = true)



In [24]:
# Preview cleaned data
df.show(5, truncate=True)

+-------+---------+--------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+----+---------+----------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|Year|Month|DayOfWeek|Hour|YearMonth|TotalPrice|
+-------+---------+--------------------+--------+-------------------+---------+----------+--------------+----+-----+---------+----+---------+----------+
| 489437|    10002|INFLATABLE POLITI...|      12|2009-12-01 09:08:00|     0.85|     15362|United Kingdom|2009|   12|        3|   9|  2009-12|      10.2|
| 489465|   72760B|VINTAGE CREAM 3 B...|       4|2009-12-01 10:52:00|     9.95|     13767|United Kingdom|2009|   12|        3|  10|  2009-12|      39.8|
| 489465|    84879|ASSORTED COLOUR B...|     160|2009-12-01 10:52:00|     1.45|     13767|United Kingdom|2009|   12|        3|  10|  2009-12|     232.0|
| 489517|    21584|RETRO SPOT SMALL ...|      20|2009-12-01 11:34:00|     1.65|   

In [25]:
# Summary stats on cleaned data
df.select("Quantity", "UnitPrice", "TotalPrice").describe().show()

# Confirm no nulls remain in key columns
print("Remaining nulls in key columns:")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["Invoice", "CustomerID", "Quantity", "UnitPrice", "TotalPrice"]
]).show()

+-------+------------------+------------------+------------------+
|summary|          Quantity|         UnitPrice|        TotalPrice|
+-------+------------------+------------------+------------------+
|  count|            779425|            779425|            779425|
|   mean|13.489369727683869| 3.218487985375475| 22.29182313884957|
| stddev| 145.8558140997767|29.676139695711434|227.42707548583434|
|    min|                 1|             0.001|               0.0|
|    max|             80995|           10953.5|          168469.6|
+-------+------------------+------------------+------------------+

Remaining nulls in key columns:


+-------+----------+--------+---------+----------+
|Invoice|CustomerID|Quantity|UnitPrice|TotalPrice|
+-------+----------+--------+---------+----------+
|      0|         0|       0|        0|         0|
+-------+----------+--------+---------+----------+



In [26]:
# Before vs After summary
cleaned_count = df.count()
print("=" * 45)
print(f"  Raw records      : {raw_count:>10,}")
print(f"  Cleaned records  : {cleaned_count:>10,}")
print(f"  Records removed  : {raw_count - cleaned_count:>10,}")
print(f"  Data retained    : {cleaned_count/raw_count*100:>9.1f}%")
print("=" * 45)

  Raw records      :  1,067,371
  Cleaned records  :    779,425
  Records removed  :    287,946
  Data retained    :      73.0%


## 8. Save Cleaned Data — HDFS & Local (for GitHub)

In [28]:
df.write.parquet(
    "hdfs://localhost:8020/retail/cleaned/",
    mode="overwrite"
)
print("Cleaned data saved to HDFS: hdfs://localhost:8020/retail/cleaned/")

26/04/26 09:57:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Cleaned data saved to HDFS: hdfs://localhost:8020/retail/cleaned/


In [29]:
# ── Save locally as CSV (for GitHub submission) ───────────────────────────────
# Writing a sample (first 50,000 rows) to keep file size manageable for git
df.limit(50000).toPandas().to_csv(
    "output/cleaned_data/retail_cleaned_sample.csv",
    index=False
)

# Save full schema info locally
schema_str = df.schema.json()
with open("output/cleaned_data/schema.json", "w") as f:
    f.write(schema_str)

print("Sample CSV saved locally: output/cleaned_data/retail_cleaned_sample.csv")
print("Schema saved locally:     output/cleaned_data/schema.json")

Sample CSV saved locally: output/cleaned_data/retail_cleaned_sample.csv
Schema saved locally:     output/cleaned_data/schema.json


In [31]:
# ── Verify HDFS save by reading back ─────────────────────────────────────────
df_check = spark.read.parquet("hdfs://localhost:8020/retail/cleaned/")
print(f"HDFS read-back count: {df_check.count():,} rows")
print(f"Columns: {df_check.columns}")
print("\nNotebook 01 complete. Other notebooks can now load from HDFS.")

HDFS read-back count: 779,425 rows
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Year', 'Month', 'DayOfWeek', 'Hour', 'YearMonth', 'TotalPrice']

Notebook 01 complete. Other notebooks can now load from HDFS.


---
## Summary

| Step | Action | Design Pattern |
|------|--------|----------------|
| 1 | Upload raw CSV to HDFS | Data Ingestion |
| 2 | Remove null CustomerID / Description | Filtering |
| 3 | Remove cancelled orders (Invoice starts with C) | Filtering |
| 4 | Remove Quantity ≤ 0, UnitPrice ≤ 0 | Filtering |
| 5 | Remove duplicate rows | Deduplication |
| 6 | Parse dates, extract Year/Month/Day/Hour | Transformation |
| 7 | Add TotalPrice = Quantity × UnitPrice | Transformation |
| 8 | Save cleaned Parquet to HDFS + local CSV | Aggregation / Output |
